In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
dir_path = '/Users/RachidAJ/Desktop/Statistics Projects'
data = pd.read_csv(f'{dir_path}/data.csv')

# 1st Approach

In [2]:
transaction_df = data.groupby('order_id')['aisle'].apply(list).reset_index(name='aisle')
print(transaction_df.head())

transactions = transaction_df['aisle'].tolist()
print("\nExample Transactions List (first 5):")
print(transactions[:5])

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True, low_memory=True,verbose = 1)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7) 

   order_id                                              aisle
0         1  [yogurt, other creams cheeses, fresh vegetable...
1         2  [eggs, fresh vegetables, spices seasonings, oi...
2         3  [yogurt, soy lactosefree, packaged vegetables ...
3         4  [breakfast bakery, cold flu allergy, energy gr...
4         5  [fresh fruits, salad dressing toppings, prepar...

Example Transactions List (first 5):
[['yogurt', 'other creams cheeses', 'fresh vegetables', 'fresh vegetables', 'canned meat seafood', 'fresh fruits', 'fresh fruits', 'packaged cheese'], ['eggs', 'fresh vegetables', 'spices seasonings', 'oils vinegars', 'baking ingredients', 'fresh vegetables', 'doughs gelatins bake mixes', 'spreads', 'packaged vegetables fruits'], ['yogurt', 'soy lactosefree', 'packaged vegetables fruits', 'packaged vegetables fruits', 'soy lactosefree', 'fresh vegetables', 'poultry counter', 'bread'], ['breakfast bakery', 'cold flu allergy', 'energy granola bars', 'breakfast bars pastries', 'br

In [3]:
print("\nFrequent Itemsets:")
print(frequent_itemsets)

print("\nAssociation Rules:")
rules.head()


Frequent Itemsets:
      support                                           itemsets
0    0.076715                               (baking ingredients)
1    0.163958                                            (bread)
2    0.068667                                 (breakfast bakery)
3    0.074770                                           (butter)
4    0.069390                                  (candy chocolate)
..        ...                                                ...
151  0.051020         (yogurt, milk, packaged vegetables fruits)
152  0.050928  (yogurt, packaged cheese, packaged vegetables ...
153  0.062319  (milk, fresh fruits, fresh vegetables, package...
154  0.067929  (fresh fruits, fresh vegetables, packaged chee...
155  0.076052  (yogurt, fresh fruits, fresh vegetables, packa...

[156 rows x 2 columns]

Association Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(canned jarred vegetables),(fresh vegetables),0.073715,0.444341,0.056155,0.761785,1.714413,1.0,0.023400,2.332590,0.449872,0.121573,0.571292,0.444081
1,(canned meals beans),(fresh vegetables),0.069799,0.444341,0.050033,0.716821,1.613222,1.0,0.019019,1.962218,0.408645,0.107805,0.490373,0.414711
2,(fresh dips tapenades),(fresh fruits),0.098128,0.556755,0.069495,0.708205,1.272023,1.0,0.014861,1.519030,0.237119,0.118716,0.341685,0.416513
3,(fresh herbs),(fresh fruits),0.093564,0.556755,0.070103,0.749254,1.345752,1.0,0.018011,1.767706,0.283441,0.120823,0.434295,0.437584
4,(fresh vegetables),(fresh fruits),0.444341,0.556755,0.318137,0.715974,1.285977,1.0,0.070748,1.560581,0.400212,0.465821,0.359213,0.643694


In [4]:
import plotly.graph_objects as go

rules["antecedents"] = rules["antecedents"].astype(str)
rules["consequents"] = rules["consequents"].astype(str)

labels = list(pd.concat([rules["antecedents"], rules["consequents"]]).unique())
label_index = {label: i for i, label in enumerate(labels)}

sources = [label_index[a] for a in rules["antecedents"]]
targets = [label_index[c] for c in rules["consequents"]]
values = rules["support"].tolist()

fig = go.Figure(go.Sankey(
    node=dict(
        pad=80,  
        thickness=50,  
        line=dict(color="black", width=1), 
        label=labels
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
))

fig.update_layout(
    title_text="Diagramme de Sankey des règles d'association",
    font_size=14, 
    width=1000,  
    height=900  
)

fig.show()


# Product Approach

In [5]:
sampled_users = data['user_id'].drop_duplicates().sample(n=10000, random_state=42)
data2 = data[data['user_id'].isin(sampled_users)]

In [6]:

transaction_df = data2.groupby('order_id')['product_name'].apply(list).reset_index(name='product_name')
print(transaction_df.head())

transactions = transaction_df['product_name'].tolist()
print("\nExample Transactions List (first 5):")
print(transactions[:5])

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.001, use_colnames=True, low_memory=True,verbose = 1)

   order_id                                       product_name
0        21  [Yuba Tofu Skin, Organic Firm Tofu, Frozen Org...
1        25  [Original Popcorn, Boomchickapop Sweet & Salty...
2        42  [Unsweetened Vanilla Almond Breeze, Pure Irish...
3        47  [Strawberries, Organic Blueberries, Raspberrie...
4        98  [Natural Spring Water, Organic Orange Juice Wi...

Example Transactions List (first 5):
[['Yuba Tofu Skin', 'Organic Firm Tofu', 'Frozen Organic Blueberries', 'Dairy-Free Cheddar Style Wedges', 'Cold Brew Coffee'], ['Original Popcorn', 'Boomchickapop Sweet & Salty Kettle Corn', 'Steamfresh Lightly Sauced Broccoli With Cheese Sauce', 'Delights Turkey Sausage Egg Whites & Cheese English Muffin', 'Ultra Thin Sliced Provolone Cheese', 'Cheese Shredded Mozzarella Reduced Fat 2%', 'All Whites 100% Egg Whites', 'Smok Cured Turkey Bacon', 'Shredded Hash Browns', 'Simply 9 White Meat Chicken & Whole Barley Recipe Dog Food', 'Craveables Four Cheese Pizza', 'Macaroni and Che

KeyboardInterrupt: 

In [ ]:
#rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5) 
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)
print("\nFrequent Itemsets:")
print(frequent_itemsets)

print("\nAssociation Rules:")
rules.head()


Frequent Itemsets:
       support                                           itemsets
0     0.001608                         (0% Fat Free Organic Milk)
1     0.003911                         (0% Greek Strained Yogurt)
2     0.001558                                          (1 Liter)
3     0.001029                               (1 Ply Paper Towels)
4     0.003080                                  (1% Low Fat Milk)
...        ...                                                ...
3958  0.001016  (Sparkling Water Grapefruit, Sparkling Lemon W...
3959  0.001004  (Sparkling Water Grapefruit, Sparkling Water B...
3960  0.001146  (Total 2% Lowfat Greek Strained Yogurt With Bl...
3961  0.001047  (Organic Baby Spinach, Bag of Organic Bananas,...
3962  0.001269  (Organic Raspberries, Bag of Organic Bananas, ...

[3963 rows x 2 columns]

Association Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Banana),(1% Low Fat Milk),0.142938,0.003080,0.001029,0.007196,2.336676,1.0,0.000588,1.004146,0.667445,0.007095,0.004129,0.170598
1,(1% Low Fat Milk),(Banana),0.003080,0.142938,0.001029,0.334000,2.336676,1.0,0.000588,1.286880,0.573809,0.007095,0.222927,0.170598
2,(1% Lowfat Milk),(Banana),0.004841,0.142938,0.001367,0.282443,1.975979,1.0,0.000675,1.194416,0.496325,0.009340,0.162771,0.146005
3,(Banana),(1% Lowfat Milk),0.142938,0.004841,0.001367,0.009566,1.975979,1.0,0.000675,1.004771,0.576297,0.009340,0.004748,0.146005
4,(100% Raw Coconut Water),(Bag of Organic Bananas),0.013366,0.122039,0.002932,0.219355,1.797419,1.0,0.001301,1.124661,0.449657,0.022132,0.110843,0.121690


In [ ]:
import plotly.graph_objects as go

rules["antecedents"] = rules["antecedents"].astype(str)
rules["consequents"] = rules["consequents"].astype(str)

labels = list(pd.concat([rules["antecedents"], rules["consequents"]]).unique())
label_index = {label: i for i, label in enumerate(labels)}

sources = [label_index[a] for a in rules["antecedents"]]
targets = [label_index[c] for c in rules["consequents"]]
values = rules["support"].tolist()

fig = go.Figure(go.Sankey(
    node=dict(
        pad=80,  
        thickness=50,  
        line=dict(color="black", width=1), 
        label=labels
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
))

fig.update_layout(
    title_text="Diagramme de Sankey des règles d'association",
    font_size=14, 
    width=1000,  
    height=900  
)

fig.show()
